# Exploratory Analysis of Bitcoin Daily Log Returns

This notebook focuses on Bitcoin daily log returns. The main forecasting task in the project is to predict the **next-day Bitcoin daily log return**.

The daily log return is calculated from daily close prices:

$$
r_t = \log\left(\frac{P_t}{P_{t-1}}\right)
$$

The one-step-ahead target is:

$$
\text{target}_t = r_{t+1}
$$

So for day \(t\), we use information available up to that day and the target is the close-to-close log return on the next day. Values are kept on the normal return scale, for example \(0.025\) means about **2.5% daily log return**.

## 1. Exploratory Data Analysis

In [ ]:
from pathlib import Path
import json
import pickle
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.options.display.float_format = "{:,.6f}".format

In [ ]:
DAILY_DATA_PATH = Path("data/BTC/BTC_USD_coinbase_spot_daily_features.csv")
METADATA_PATH = Path("data/BTC/BTC_USD_coinbase_spot_daily_features.metadata.json")
SCALER_PATH = Path("data/BTC/BTC_USD_coinbase_spot_daily_features.scaler.pkl")
RAW_5MIN_PATH = Path("data/BTC/BTC_USD_coinbase_spot_5min.csv")

### Load Daily Features

The daily feature file is used mainly for daily OHLCV data and the existing train/test split. Some numeric columns in this file are scaled, so we invert the scaler before using prices and returns.

In [ ]:
daily_scaled = pd.read_csv(DAILY_DATA_PATH, parse_dates=["timestamp_utc"])

with METADATA_PATH.open("r", encoding="utf-8") as file:
    metadata = json.load(file)

with SCALER_PATH.open("rb") as file:
    scaler = pickle.load(file)

scaled_columns = metadata["scaled_columns"]
daily_full = daily_scaled.copy()
daily_full[scaled_columns] = scaler.inverse_transform(daily_scaled[scaled_columns])

daily_full = daily_full.set_index("timestamp_utc").sort_index()
daily_full.head()

### Build Daily Log Return Target from Prices

The daily feature file already contains daily close-to-close log returns and the next-day target. Here we rebuild returns from the same filtered daily close series as a consistency check and keep intraday realized volatility only as an auxiliary diagnostic for volatility clustering.

In [ ]:
raw_5min = pd.read_csv(RAW_5MIN_PATH, parse_dates=["timestamp_utc"])
raw_5min = raw_5min.sort_values("timestamp_utc").set_index("timestamp_utc")

raw_5min["log_close_5m"] = np.log(raw_5min["close"])
raw_5min["intraday_log_return"] = raw_5min["log_close_5m"].diff()
raw_5min["intraday_squared_return"] = raw_5min["intraday_log_return"] ** 2

raw_daily = raw_5min.resample("1D", closed="left", label="left").agg(
    raw_close=("close", "last"),
    intraday_observations=("intraday_log_return", "count"),
    realized_variance=("intraday_squared_return", "sum"),
)
raw_daily["realized_volatility"] = raw_daily["realized_variance"].pow(0.5)
raw_daily["realized_volatility_pct"] = raw_daily["realized_volatility"] * 100

raw_daily[["raw_close", "realized_volatility", "intraday_observations"]].head()

In [ ]:
daily_full = daily_full.join(
    raw_daily[["raw_close", "intraday_observations", "realized_variance", "realized_volatility", "realized_volatility_pct"]],
    how="left",
)

daily_full["log_return_rebuilt"] = np.log(daily_full["close"] / daily_full["close"].shift(1))
daily_full["target_next_daily_log_return"] = daily_full["log_return"].shift(-1)
daily_full["log_return_pct"] = daily_full["log_return"] * 100
daily_full["target_next_daily_log_return_pct"] = daily_full["target_next_daily_log_return"] * 100
daily_full["abs_log_return"] = daily_full["log_return"].abs()
daily_full["abs_log_return_pct"] = daily_full["abs_log_return"] * 100
daily_full["squared_log_return"] = daily_full["log_return"] ** 2
daily_full["direction"] = np.where(daily_full["log_return"] > 0, "positive", "negative")

return_check = (daily_full["log_return"] - daily_full["log_return_rebuilt"]).abs().dropna()
target_check = (daily_full["target_log_return_next"] - daily_full["target_next_daily_log_return"]).abs().dropna()
print(f"Maximum absolute rebuilt-return difference: {return_check.max():.12f}")
print(f"Maximum absolute target-shift difference: {target_check.max():.12f}")

daily_full[[
    "close",
    "raw_close",
    "log_return",
    "log_return_pct",
    "target_next_daily_log_return",
    "target_next_daily_log_return_pct",
    "split",
]].head()

### Final Forecasting Setup

For the later models: use data up to day `t` -> predict `r_{t+1}`.

`r_t` is the daily close-to-close log return. Intraday realized volatility is kept in this notebook only as supporting evidence about clustering and changing risk regimes, not as the main target.

### Dataset Overview

The EDA below uses only the training period. The test period is kept separate for later model evaluation.

In [ ]:
train_daily = daily_full.loc[daily_full["split"] == "train"].copy()
test_daily = daily_full.loc[daily_full["split"] == "test"].copy()

# Use only the training split for exploratory analysis and statistical diagnostics.
daily = train_daily.copy()

print(f"Full rows: {len(daily_full):,}")
print(f"Training rows used below: {len(daily):,}")
print(f"Test rows held out: {len(test_daily):,}")
print(f"Columns: {daily_full.shape[1]}")
print(f"Training start date: {daily.index.min().date()}")
print(f"Training end date: {daily.index.max().date()}")
print(f"Test start date: {test_daily.index.min().date()}")
print(f"Test end date: {test_daily.index.max().date()}")
print(f"Training missing values: {daily.isna().sum().sum()}")
print(f"Duplicated dates in full data: {daily_full.index.duplicated().sum()}")

In [ ]:
daily[[
    "open",
    "high",
    "low",
    "close",
    "volume",
    "log_return",
    "log_return_pct",
    "target_next_daily_log_return",
    "split",
]].head()

### Missing Values and Daily Coverage

In [ ]:
missing_values = daily.isna().sum().to_frame("missing_values")
missing_values["missing_share"] = missing_values["missing_values"] / len(daily)

missing_values[missing_values["missing_values"] > 0]

In [ ]:
daily[[
    "observations",
    "expected_observations",
    "coverage_ratio",
    "intraday_observations",
]].describe().T

In [ ]:
date_gaps = daily.index.to_series().diff()
date_gaps.value_counts().head(10)

In [ ]:
long_gaps = date_gaps[date_gaps > pd.Timedelta(days=1)]

gap_details = pd.DataFrame({
    "previous_date": [daily.index[daily.index.get_loc(date) - 1] for date in long_gaps.index],
    "current_date": long_gaps.index,
    "gap_length": long_gaps.values,
})

gap_details.head(30)

### Summary Statistics

In [ ]:
main_columns = [
    "open",
    "high",
    "low",
    "close",
    "volume",
    "log_return",
    "log_return_pct",
    "abs_log_return_pct",
    "squared_log_return",
    "target_next_daily_log_return",
    "target_next_daily_log_return_pct",
]

daily[main_columns].describe().T

In [ ]:
largest_positive_return_date = daily["log_return"].idxmax()
largest_negative_return_date = daily["log_return"].idxmin()
largest_absolute_return_date = daily["abs_log_return"].idxmax()

print("Largest positive daily log return:")
print(largest_positive_return_date.date(), f"{daily.loc[largest_positive_return_date, 'log_return_pct']:.2f}%")

print("\nLargest negative daily log return:")
print(largest_negative_return_date.date(), f"{daily.loc[largest_negative_return_date, 'log_return_pct']:.2f}%")

print("\nLargest absolute daily log return:")
print(largest_absolute_return_date.date(), f"{daily.loc[largest_absolute_return_date, 'abs_log_return_pct']:.2f}%")

### Price, Volume, and Daily Log Returns

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].plot(daily.index, daily["close"], color="tab:blue")
axes[0].set_title("BTC daily closing price")
axes[0].set_ylabel("USD")

axes[1].plot(daily.index, daily["volume"], color="tab:green")
axes[1].set_title("Daily trading volume")
axes[1].set_ylabel("Volume")

axes[2].plot(daily.index, daily["log_return_pct"], color="tab:gray", linewidth=0.8)
axes[2].axhline(0, color="black", linewidth=0.8)
axes[2].set_title("Daily close-to-close log return")
axes[2].set_ylabel("Percent")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(daily["log_return_pct"].dropna(), bins=100, kde=True, ax=axes[0], color="tab:blue")
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_title("Distribution of daily log returns")
axes[0].set_xlabel("Daily log return, percent")

sns.histplot(daily["abs_log_return_pct"].dropna(), bins=100, kde=True, ax=axes[1], color="tab:red")
axes[1].set_title("Distribution of absolute daily log returns")
axes[1].set_xlabel("Absolute daily log return, percent")

plt.tight_layout()
plt.show()

## 2. Trend, Seasonality, and Volatility

This section checks how daily log returns behave over time. We look at the return level, rolling return volatility, calendar patterns, and volatility clustering in absolute and squared returns.

### Trend

In [ ]:
daily["return_30d_mean"] = daily["log_return"].rolling(30).mean()
daily["return_90d_mean"] = daily["log_return"].rolling(90).mean()
daily["return_30d_volatility"] = daily["log_return"].rolling(30).std()

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(daily.index, daily["log_return_pct"], label="Daily log return", linewidth=0.7, alpha=0.55)
axes[0].plot(daily.index, daily["return_30d_mean"] * 100, label="30-day mean", linewidth=2)
axes[0].plot(daily.index, daily["return_90d_mean"] * 100, label="90-day mean", linewidth=2)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("BTC daily log returns over time")
axes[0].set_ylabel("Return, percent")
axes[0].legend()

axes[1].plot(daily.index, daily["return_30d_volatility"] * 100, color="tab:red", linewidth=1.2)
axes[1].set_title("30-day rolling volatility of daily log returns")
axes[1].set_ylabel("Standard deviation, percent")

plt.tight_layout()
plt.show()

In [ ]:
return_trend_data = daily["log_return"].dropna()
x = np.arange(len(return_trend_data))
trend_coef = np.polyfit(x, return_trend_data, 1)
trend_line = np.polyval(trend_coef, x)

plt.figure(figsize=(12, 4))
plt.plot(return_trend_data.index, return_trend_data * 100, label="Daily log return", linewidth=0.7, alpha=0.55)
plt.plot(return_trend_data.index, trend_line * 100, label="Linear trend", linewidth=2)
plt.axhline(0, color="black", linewidth=0.8)
plt.title("BTC daily log returns with linear trend")
plt.ylabel("Log return, percent")
plt.legend()
plt.tight_layout()
plt.show()

### Seasonality

In [ ]:
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

daily["weekday"] = daily.index.day_name()
daily["month"] = daily.index.month_name()

weekday_summary = daily.groupby("weekday")["log_return"].agg(
    days="count",
    mean_return="mean",
    median_return="median",
    return_volatility="std",
    mean_absolute_return=lambda x: x.abs().mean(),
    positive_share=lambda x: (x > 0).mean(),
).reindex(weekday_order)

weekday_summary.assign(
    mean_return_pct=weekday_summary["mean_return"] * 100,
    median_return_pct=weekday_summary["median_return"] * 100,
    return_volatility_pct=weekday_summary["return_volatility"] * 100,
    mean_absolute_return_pct=weekday_summary["mean_absolute_return"] * 100,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

weekday_plot = weekday_summary.reset_index()
weekday_plot["mean_return_pct"] = weekday_plot["mean_return"] * 100
weekday_plot["return_volatility_pct"] = weekday_plot["return_volatility"] * 100

sns.barplot(
    data=weekday_plot,
    x="weekday",
    y="mean_return_pct",
    ax=axes[0],
    color="tab:blue",
)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("Average daily log return by weekday")
axes[0].set_xlabel("")
axes[0].set_ylabel("Mean log return, percent")
axes[0].tick_params(axis="x", rotation=45)

sns.barplot(
    data=weekday_plot,
    x="weekday",
    y="return_volatility_pct",
    ax=axes[1],
    color="tab:red",
)
axes[1].set_title("Return volatility by weekday")
axes[1].set_xlabel("")
axes[1].set_ylabel("Standard deviation, percent")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
month_order = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December",
]

monthly_summary = daily.groupby("month")["log_return"].agg(
    days="count",
    mean_return="mean",
    median_return="median",
    return_volatility="std",
    mean_absolute_return=lambda x: x.abs().mean(),
    positive_share=lambda x: (x > 0).mean(),
).reindex(month_order)

monthly_summary.assign(
    mean_return_pct=monthly_summary["mean_return"] * 100,
    median_return_pct=monthly_summary["median_return"] * 100,
    return_volatility_pct=monthly_summary["return_volatility"] * 100,
    mean_absolute_return_pct=monthly_summary["mean_absolute_return"] * 100,
)

In [ ]:
monthly_plot = monthly_summary.reset_index()
monthly_plot["mean_return_pct"] = monthly_plot["mean_return"] * 100
monthly_plot["return_volatility_pct"] = monthly_plot["return_volatility"] * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.barplot(
    data=monthly_plot,
    x="month",
    y="mean_return_pct",
    color="tab:green",
    ax=axes[0],
)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("Average daily log return by month")
axes[0].set_xlabel("")
axes[0].set_ylabel("Mean log return, percent")
axes[0].tick_params(axis="x", rotation=45)

sns.barplot(
    data=monthly_plot,
    x="month",
    y="return_volatility_pct",
    color="tab:red",
    ax=axes[1],
)
axes[1].set_title("Return volatility by month")
axes[1].set_xlabel("")
axes[1].set_ylabel("Standard deviation, percent")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

### Volatility Clustering

In [ ]:
daily["abs_log_return_lag_1"] = daily["abs_log_return"].shift(1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(
    daily["abs_log_return_lag_1"] * 100,
    daily["abs_log_return"] * 100,
    alpha=0.35,
    s=12,
)
axes[0].set_title("Absolute log return vs previous day")
axes[0].set_xlabel("Previous-day absolute log return, percent")
axes[0].set_ylabel("Current absolute log return, percent")

quartile_data = daily.dropna(subset=["abs_log_return_lag_1", "abs_log_return_pct"]).copy()
quartile_data["previous_abs_return_quartile"] = pd.qcut(
    quartile_data["abs_log_return_lag_1"],
    4,
    labels=["Q1", "Q2", "Q3", "Q4"],
)

sns.boxplot(
    data=quartile_data,
    x="previous_abs_return_quartile",
    y="abs_log_return_pct",
    ax=axes[1],
    color="tab:red",
)
axes[1].set_title("Current absolute return by previous-day quartile")
axes[1].set_xlabel("Previous-day absolute return quartile")
axes[1].set_ylabel("Current absolute log return, percent")

plt.tight_layout()
plt.show()

In [ ]:
largest_return_moves = daily.sort_values("abs_log_return", ascending=False)[
    ["close", "log_return", "log_return_pct", "abs_log_return_pct", "volume", "realized_volatility_pct"]
].head(10)

largest_return_moves

## 3. Stationarity and Autocorrelation

This section checks whether daily log returns are stable enough to model and whether past returns, absolute returns, or squared returns are related to future values.

### Stationarity Tests

In [ ]:
def stationarity_tests(series, series_name):
    clean_series = series.dropna()
    adf_result = adfuller(clean_series, autolag="AIC")
    kpss_result = kpss(clean_series, regression="c", nlags="auto")

    return pd.Series({
        "series": series_name,
        "observations": len(clean_series),
        "adf_statistic": adf_result[0],
        "adf_pvalue": adf_result[1],
        "kpss_statistic": kpss_result[0],
        "kpss_pvalue": kpss_result[1],
    })

In [ ]:
stationarity_summary = pd.DataFrame([
    stationarity_tests(daily["close"], "close price"),
    stationarity_tests(daily["log_close"], "log close price"),
    stationarity_tests(daily["log_return"], "daily log return"),
    stationarity_tests(daily["abs_log_return"], "absolute daily log return"),
    stationarity_tests(daily["squared_log_return"], "squared daily log return"),
    stationarity_tests(daily["realized_volatility"], "realized volatility, auxiliary"),
])

stationarity_summary

The stationarity tests show that the BTC close price and log close price are non-stationary. Their ADF p-values are high, so we fail to reject the presence of a unit root, while the KPSS p-values are low, so we reject stationarity.

Daily log returns appear much more suitable for time-series modeling. The ADF test rejects a unit root, and the KPSS result is typically less problematic than for price levels.

Absolute and squared log returns describe the size of market moves rather than their direction. They are useful diagnostics for volatility clustering: even when raw returns have weak autocorrelation, the magnitude of returns can remain persistent.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(daily.index, daily["log_close"], color="tab:blue")
axes[0].set_title("Log close price")
axes[0].set_ylabel("Log price")

axes[1].plot(daily.index, daily["log_return_pct"], color="tab:gray", linewidth=0.8)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Daily log return")
axes[1].set_ylabel("Percent")

axes[2].plot(daily.index, daily["abs_log_return_pct"], color="tab:red", linewidth=0.8)
axes[2].set_title("Absolute daily log return")
axes[2].set_ylabel("Percent")

plt.tight_layout()
plt.show()

### Autocorrelation Plots

In [ ]:
returns = daily["log_return"].dropna()
squared_returns = daily["squared_log_return"].dropna()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
plot_acf(returns, lags=40, ax=axes[0, 0])
plot_pacf(returns, lags=40, ax=axes[0, 1], method="ywm")
plot_acf(squared_returns, lags=40, ax=axes[1, 0])
plot_pacf(squared_returns, lags=40, ax=axes[1, 1], method="ywm")

axes[0, 0].set_title("ACF of daily log returns")
axes[0, 1].set_title("PACF of daily log returns")
axes[1, 0].set_title("ACF of squared daily log returns")
axes[1, 1].set_title("PACF of squared daily log returns")

plt.tight_layout()
plt.show()

In [ ]:
absolute_returns = daily["abs_log_return"].dropna()
realized_vol = daily["realized_volatility"].dropna()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(absolute_returns, lags=40, ax=axes[0])
plot_acf(realized_vol, lags=40, ax=axes[1])
axes[0].set_title("ACF of absolute daily log returns")
axes[1].set_title("ACF of realized volatility (auxiliary)")
plt.tight_layout()
plt.show()

### Ljung-Box Tests

In [ ]:
ljung_lags = [5, 10, 20]

ljung_inputs = {
    "daily log return": returns,
    "absolute daily log return": absolute_returns,
    "squared daily log return": squared_returns,
    "realized volatility, auxiliary": realized_vol,
}

ljung_rows = []
for series_name, series_values in ljung_inputs.items():
    result = acorr_ljungbox(series_values, lags=ljung_lags, return_df=True)
    result["series"] = series_name
    result.index.name = "lag"
    ljung_rows.append(result.reset_index())

ljung_box_summary = pd.concat(ljung_rows, ignore_index=True)[
    ["series", "lag", "lb_stat", "lb_pvalue"]
]

ljung_box_summary

In [ ]:
ljung_box_summary.assign(significant=ljung_box_summary["lb_pvalue"] < 0.05)

The Ljung-Box test separates two parts of the return process. Raw daily log returns have limited autocorrelation, which means the conditional mean is difficult to forecast from simple linear dependence alone.

Absolute and squared daily log returns show stronger dependence. This indicates volatility clustering: large market moves tend to be followed by periods of larger moves, and calm periods tend to persist.

Realized volatility, kept here as an auxiliary intraday-based measure, also shows persistence. This supports the hybrid project design: the LSTM targets the next daily log return, while the SGARCH component is motivated by persistent volatility in the residual or return magnitude process.